### Importando bibliotecas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from collections import Counter
import time

### Dados

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
microdados = pd.read_parquet('/content/drive/MyDrive/PIBIC/Dados/dadosCenso2018_featuresML.parquet')

In [ ]:
microdados.head()

In [ ]:
microdados.describe()

### Machine Learning

In [ ]:
!pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

In [ ]:
microdados['TP_SITUACAO'].unique()

#### Versão Original

In [ ]:
# --- PASSO 1: Filtragem e Labeling ---
# Mapeamento: 4 (Desvinculado) e 5 (Transferido) = EVASÃO (1)
# Mapeamento: 6 (Formado) = SUCESSO (0)
# Ignoramos 7 (Falecido) e separamos 2 e 3 para predição futura.

df_treino = microdados[microdados['TP_SITUACAO'].isin([4, 5, 6])].copy()
df_treino['TARGET'] = df_treino['TP_SITUACAO'].apply(lambda x: 1 if x in [4, 5] else 0)

In [ ]:
# --- PASSO 2: Seleção de Atributos Sugeridos (Privacidade Garantida) ---
features = [
  'NU_DIA_NASCIMENTO', 'NU_MES_NASCIMENTO', 'NU_ANO_NASCIMENTO', 'TP_SEXO', 'TP_COR_RACA',
  'CO_MUNICIPIO_NASCIMENTO', 'TP_NACIONALIDADE', 'CO_PAIS_ORIGEM', 'CO_IES', 'CO_CURSO', 'TP_ESCOLA_CONCLUSAO_ENS_MEDIO'
]

X = df_treino[features]
y = df_treino['TARGET']

In [ ]:
# --- PASSO 3: Transformação de Categorias ---
# XGBoost exige que categorias sejam numéricas
le = LabelEncoder()
for col in X.columns:
    X[col] = le.fit_transform(X[col].astype(str))

In [ ]:
# --- PASSO 4: Divisão e Treinamento ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Configuração simples e eficiente do XGBoost
model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    verbosity=1
)

model.fit(X_train, y_train)

In [ ]:
# --- PASSO 5: Avaliação ---
print("Relatório de Classificação (Evasão vs Formados):")
print(classification_report(y_test, model.predict(X_test)))

#### Versão Modificada 1

In [ ]:
# --- PASSO 1: Filtragem e Labeling ---
# Mapeamento: 4 (Desvinculado) e 5 (Transferido) = EVASÃO (1)
# Mapeamento: 6 (Formado) = SUCESSO (0)
# Ignoramos 7 (Falecido) e separamos 2 e 3 para predição futura.

df_treino = microdados[microdados['TP_SITUACAO'].isin([4, 5, 6])].copy()
df_treino['TARGET'] = df_treino['TP_SITUACAO'].apply(lambda x: 1 if x in [4, 5] else 0)

In [ ]:
# --- PASSO 2: Seleção de Atributos Sugeridos (Privacidade Garantida) ---
features = [
    'CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO',
    'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE',
    'CO_PAIS_ORIGEM', 'CO_CINE_AREA_GERAL'
]

X = df_treino[features]
y = df_treino['TARGET']

In [ ]:
# --- PASSO 3: Transformação de Categorias ---
# XGBoost exige que categorias sejam numéricas
le = LabelEncoder()
for col in X.columns:
    X[col] = le.fit_transform(X[col].astype(str))

In [ ]:
# --- PASSO 4: Divisão e Treinamento ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Configuração simples e eficiente do XGBoost
model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    verbosity=1
)

model.fit(X_train, y_train)

In [ ]:
# --- PASSO 5: Avaliação ---
print("Relatório de Classificação (Evasão vs Formados):")
print(classification_report(y_test, model.predict(X_test)))

In [ ]:
# --- PASSO 1: Preparação dos Dados ---
df_treino = microdados[microdados['TP_SITUACAO'].isin([4, 5, 6])].copy()
# Alvo: 1 (Evasão/Transferência) e 0 (Formado)
df_treino['TARGET'] = df_treino['TP_SITUACAO'].apply(lambda x: 1 if x in [4, 5] else 0)

features = [
    'CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO',
    'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE',
    'CO_PAIS_ORIGEM', 'CO_CINE_AREA_GERAL'
]

X = df_treino[features].copy()
y = df_treino['TARGET']

# --- PASSO 2: Transformação de Categorias ---
le = LabelEncoder()
for col in X.columns:
    X[col] = le.fit_transform(X[col].astype(str))

# --- PASSO 3: Divisão Estratificada ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# --- PASSO 4: Cálculo do Peso da Classe (Balanceamento) ---
# Isso ajuda o modelo a dar mais importância para a Classe 1 (Evasão)
count_class_0 = (y_train == 0).sum()
count_class_1 = (y_train == 1).sum()
scale_weight = count_class_0 / count_class_1

# --- PASSO 5: Configuração Otimizada do XGBoost ---
model = XGBClassifier(
    n_estimators=500,           # Mais árvores, mas usaremos early stopping
    max_depth=7,                # Profundidade maior para captar relações complexas
    learning_rate=0.05,         # Taxa menor para um aprendizado mais robusto
    scale_pos_weight=scale_weight,
    tree_method='hist',         # Otimização para grandes volumes de dados
    random_state=42,
    verbosity=1
)

# Treinamento com monitoramento (evita treinar 500 árvores se o erro parar de cair)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    early_stopping_rounds=20,
    verbose=False
)

# --- PASSO 6: Avaliação ---
y_pred = model.predict(X_test)

print("Relatório de Classificação Atualizado:")
print(classification_report(y_test, y_pred))

#### Versão Modificada 2

In [ ]:
# --- PASSO 1: Filtragem e Labeling ---
# Mapeamento: 4 (Desvinculado) e 5 (Transferido) = EVASÃO (1)
# Mapeamento: 6 (Formado) = SUCESSO (0)
# Ignoramos 7 (Falecido) e separamos 2 e 3 para predição futura.

df_treino = microdados[microdados['TP_SITUACAO'].isin([4, 5, 6])].copy()
df_treino['TARGET'] = df_treino['TP_SITUACAO'].apply(lambda x: 1 if x in [4, 5] else 0)

In [ ]:
# --- PASSO 2: Seleção de Atributos Sugeridos (Privacidade Garantida) ---
features = [
    'CO_UF', 'FAIXA_IBGE', 'TP_COR_RACA', 'TP_SEXO',
    'TP_ESCOLA_CONCLUSAO_ENS_MEDIO', 'TP_NACIONALIDADE',
    'CO_PAIS_ORIGEM', 'CO_CINE_ROTULO'
]

X = df_treino[features]
y = df_treino['TARGET']

In [ ]:
# --- PASSO 3: Transformação de Categorias ---
# XGBoost exige que categorias sejam numéricas
le = LabelEncoder()
for col in X.columns:
    X[col] = le.fit_transform(X[col].astype(str))

In [ ]:
# --- PASSO 4: Divisão e Treinamento ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Configuração simples e eficiente do XGBoost
model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    verbosity=1
)

model.fit(X_train, y_train)

In [ ]:
# --- PASSO 5: Avaliação ---
print("Relatório de Classificação (Evasão vs Formados):")
print(classification_report(y_test, model.predict(X_test)))